# AksaraLine 1/2 — build the glyph pool and render the corpus

CPU only. Writes everything to `/kaggle/working/build`, which Kaggle saves as
this kernel's output; the training kernel then attaches that output instead of
re-rendering. Splitting the two matters because a GPU session is capped at 9h
and `/kaggle/working` does not survive between runs — folding rendering into
the training kernel would repeat ~1h of CPU work every time and risk losing the
whole session to a timeout.


In [ ]:
import os, sys, subprocess, zipfile, time
from pathlib import Path

BRANCH  = 'aksara-seq'
REPO    = Path('/kaggle/working/aksara_OCR')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/phoenixfin/aksantara-ocr.git',
                    str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'aksara_seq' / 'src'))
print('commit:', subprocess.run(['git','rev-parse','--short','HEAD'],
      capture_output=True, text=True).stdout.strip())


In [ ]:
DATASET   = Path('/kaggle/input/aksara-clean-4')
DATA_ROOT = Path('/kaggle/working/data/clean')
SCRIPTS   = ['Sunda', 'Jawa', 'Bali', 'Lontara']

# One zip per script. Kaggle auto-extracts archives inconsistently, so handle
# both the extracted and the still-zipped layout rather than assuming one.
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for s in SCRIPTS:
    if (DATA_ROOT / s).is_dir():
        continue
    if (DATASET / s).is_dir():
        os.symlink(DATASET / s, DATA_ROOT / s)
        continue
    zf = DATASET / f'{s}.zip'
    assert zf.exists(), f'neither {DATASET/s} nor {zf} found'
    t0 = time.time()
    with zipfile.ZipFile(zf) as z:
        z.extractall(DATA_ROOT)
    print(f'{s}: extracted in {time.time()-t0:.0f}s')

for s in SCRIPTS:
    print(f'  {s:9s} {sum(1 for p in (DATA_ROOT/s).rglob("*") if p.is_file())} files')


## Glyph pool

Verifies each (onset × vowel) grid against its known class count before writing.


In [ ]:
BUILD  = Path('/kaggle/working/build')
GLYPHS = BUILD / 'glyphs'
!python aksara_seq/scripts/01_build_glyph_pool.py     --data-root {DATA_ROOT} --out {GLYPHS} --workers 4 --no-hash


## Render, then verify

The verifier re-derives split disjointness from the written labels rather than
trusting the generator, and checks every bounding box.


In [ ]:
CORPUS = BUILD / 'corpus' / 'v1'
!python aksara_seq/scripts/02_render_corpus.py     --config aksara_seq/configs/corpus_v1.yaml     --glyph-cache {GLYPHS} --out {CORPUS}
!python aksara_seq/scripts/03_verify_corpus.py --corpus {CORPUS}


## Trim the output

The glyph cache is ~500 MB of intermediate that the trainer never reads —
dropping it keeps this kernel's output small enough to attach comfortably.


In [ ]:
import shutil
shutil.rmtree(GLYPHS, ignore_errors=True)
shutil.rmtree('/kaggle/working/data', ignore_errors=True)
shutil.rmtree(REPO, ignore_errors=True)
total = sum(p.stat().st_size for p in BUILD.rglob('*') if p.is_file())
print(f'output: {total/1e9:.2f} GB')
